# 05 — Model Evaluation & Analysis

**Project:** `ecommerce-delivery-delay-prediction`  
**Phase:** 5 — Evaluation & Model Analysis

## Candidate

Phase 4B selected:

`xgboost_03`

using the highest mean PR-AUC across expanding-window temporal validation folds.

## Evaluation policy

The original Phase 3 test set was already observed during Phase 4A and that observation motivated the Phase 4B tuning exercise.

Therefore this notebook uses the test set as an:

**observed holdout diagnostic**

—not as a pristine unbiased final holdout estimate.

No model family, hyperparameter or feature decision should be changed because of the Phase 5 test result.

## Goals

- evaluate OOF and observed-holdout ranking quality;
- normalize PR-AUC by prevalence;
- inspect temporal robustness;
- analyze threshold/business trade-offs;
- inspect calibration;
- analyze errors across business segments;
- measure development → test feature drift;
- inspect XGBoost native importance;
- compute raw-feature permutation importance;
- compute SHAP global importance;
- produce artifacts suitable for a model card and portfolio README.

## 1. Optional SHAP dependency

In [1]:
# Run once if SHAP is not installed:
#
# uv add "shap>=0.52.0"
#
# SHAP 0.52 supports current Python 3.14 wheels.

try:
    import shap
    print("shap:", shap.__version__)
except ImportError:
    print(
        "SHAP is not installed. "
        "Run: uv add 'shap>=0.52.0'"
    )

shap: 0.52.0


## 2. Imports and paths

In [2]:
from pathlib import Path
import json
import sys

import joblib
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
from IPython.display import display

pd.set_option("display.max_columns", 200)
pd.set_option("display.max_rows", 200)
pd.set_option("display.float_format", lambda x: f"{x:,.4f}")

def find_project_root(start: Path | None = None) -> Path:
    current = (start or Path.cwd()).resolve()

    for candidate in [current, *current.parents]:
        if (candidate / "models").exists():
            return candidate

    raise FileNotFoundError("Project root not found.")

PROJECT_ROOT = find_project_root()

PROCESSED_DIR = PROJECT_ROOT / "data" / "processed"
MODELS_DIR = PROJECT_ROOT / "models"
METRICS_DIR = PROJECT_ROOT / "reports" / "metrics"
FIGURES_DIR = PROJECT_ROOT / "reports" / "figures"

if str(PROJECT_ROOT) not in sys.path:
    sys.path.insert(0, str(PROJECT_ROOT))

print("PROJECT_ROOT:", PROJECT_ROOT)

PROJECT_ROOT: /home/namdp/Documents/Projects/ecommerce-delivery-delay-prediction


## 3. Load reusable analysis utilities

In [3]:
from src.features.build_features import (
    CATEGORICAL_FEATURES,
    NUMERIC_FEATURES,
    TARGET_COLUMN,
)
from src.features.preprocessing import select_model_matrix
from src.models.analysis_utils import (
    add_analysis_segments,
    calibration_table,
    ranking_metrics,
    segment_metrics,
    threshold_metrics,
    threshold_sweep,
)
from src.models.drift import drift_summary
from src.models.evaluate import predict_positive_probability
from src.models.explain import (
    native_xgboost_importance,
    raw_permutation_importance,
    shap_global_importance,
)

## 4. Load selected candidate and data

In [4]:
model = joblib.load(
    MODELS_DIR / "best_tuned_candidate.joblib"
)

with (
    MODELS_DIR / "best_tuned_candidate_metadata.json"
).open(
    "r",
    encoding="utf-8",
) as file:
    candidate_metadata = json.load(file)

train_df = pd.read_parquet(
    PROCESSED_DIR / "train.parquet"
)
validation_df = pd.read_parquet(
    PROCESSED_DIR / "validation.parquet"
)
test_df = pd.read_parquet(
    PROCESSED_DIR / "test.parquet"
)

development_df = (
    pd.concat(
        [train_df, validation_df],
        ignore_index=True,
    )
    .sort_values(
        ["prediction_timestamp", "order_id"],
        kind="stable",
    )
    .reset_index(drop=True)
)

oof_df = pd.read_parquet(
    METRICS_DIR / "best_tuned_oof_predictions.parquet"
)

print("Candidate:", candidate_metadata["best_config_id"])
print("Family:", candidate_metadata["model_family"])
print("Development:", development_df.shape)
print("OOF:", oof_df.shape)
print("Observed test:", test_df.shape)

Candidate: xgboost_03
Family: xgboost
Development: (81982, 44)
OOF: (40991, 5)
Observed test: (14468, 44)


## 5. Score the observed holdout

In [5]:
X_test = select_model_matrix(test_df)
y_test = test_df[TARGET_COLUMN].copy()

test_probability = predict_positive_probability(
    model,
    X_test,
)

test_scored = test_df.copy()
test_scored["probability"] = test_probability
test_scored = add_analysis_segments(test_scored)

assert len(test_probability) == len(test_df)
assert np.isfinite(test_probability).all()
assert ((test_probability >= 0) & (test_probability <= 1)).all()

print("✓ Observed holdout scored.")

✓ Observed holdout scored.


## 6. Ranking quality

PR-AUC is reported together with positive prevalence.

We also report:

`PR-AUC lift = PR-AUC / prevalence`

because raw PR-AUC can change substantially when the late-delivery rate changes across time.

In [6]:
ranking_df = pd.DataFrame(
    [
        {
            "dataset": "development_oof",
            "status": "temporal_oof",
            **ranking_metrics(
                oof_df["late_delivery"],
                oof_df["probability"],
            ),
        },
        {
            "dataset": "phase4a_test",
            "status": "observed_holdout_diagnostic",
            **ranking_metrics(
                y_test,
                test_probability,
            ),
        },
    ]
)

display(ranking_df)

,dataset,status,rows,positives,prevalence,pr_auc,pr_auc_lift,roc_auc,brier_score,log_loss
0,development_oof,temporal_oof,40991,4132,0.1008,0.1907,1.8914,0.6919,0.2015,0.5888
1,phase4a_test,observed_holdout_diagnostic,14468,952,0.0658,0.1080,1.6410,0.6796,0.1190,0.3957


## 7. Temporal robustness of selected configuration

In [7]:
fold_metrics = pd.read_csv(
    METRICS_DIR / "temporal_tuning_fold_metrics.csv"
)

selected_folds = (
    fold_metrics[
        fold_metrics["config_id"]
        == candidate_metadata["best_config_id"]
    ]
    .copy()
    .sort_values("fold")
)

selected_folds["pr_auc_lift"] = (
    selected_folds["pr_auc"]
    / selected_folds["validation_late_rate"]
)

display(
    selected_folds[
        [
            "fold",
            "train_rows",
            "validation_rows",
            "train_late_rate",
            "validation_late_rate",
            "pr_auc",
            "pr_auc_lift",
            "roc_auc",
            "precision",
            "recall",
            "f1",
        ]
    ]
)

,fold,train_rows,validation_rows,train_late_rate,validation_late_rate,pr_auc,pr_auc_lift,roc_auc,precision,recall,f1
64,1,40991,10248,0.0668,0.0625,0.1052,1.6825,0.6420,0.1075,0.3167,0.1605
65,2,51239,10248,0.0660,0.2074,0.3738,1.8027,0.6926,0.3432,0.5675,0.4277
66,3,61487,10248,0.0895,0.0754,0.2499,3.3128,0.7697,0.1038,0.8991,0.1861
67,4,71735,10247,0.0875,0.0579,0.1778,3.0717,0.7749,0.2286,0.2698,0.2475


### Interpretation

Use both raw PR-AUC and PR-AUC lift.

A fold with a much higher positive prevalence naturally has a higher no-skill PR-AUC baseline, so raw PR-AUC values should not be compared without context.

## 8. Threshold / business trade-off

In [8]:
thresholds = pd.read_csv(
    METRICS_DIR / "tuned_operating_thresholds.csv"
)

threshold_lookup = {
    str(row["threshold_rule"]): float(row["threshold"])
    for _, row in thresholds.iterrows()
}

threshold_results = []

for name, threshold in {
    "default_0.50": 0.50,
    **threshold_lookup,
}.items():
    threshold_results.append(
        {
            "rule": name,
            **threshold_metrics(
                y_test,
                test_probability,
                threshold,
            ),
        }
    )

threshold_results = pd.DataFrame(threshold_results)
display(threshold_results)

,rule,threshold,precision,recall,f1,positive_rate,tp,fp,tn,fn
0,default_0.50,0.5000,0.1121,0.1597,0.1317,0.0937,152,1204,12312,800
1,best_oof_f1,0.5650,0.1073,0.0914,0.0987,0.0561,87,724,12792,865
2,maximize_precision_subject_to_recall>=0.50,0.5260,0.1027,0.1197,0.1106,0.0767,114,996,12520,838


### 8.1 Full threshold sweep

In [9]:
threshold_sweep_df = threshold_sweep(
    y_test,
    test_probability,
)

display(threshold_sweep_df)

,threshold,precision,recall,f1,positive_rate,tp,fp,tn,fn
0,0.0500,0.0668,1.0000,0.1253,0.9843,952,13289,227,0
1,0.1000,0.0714,0.9895,0.1331,0.9122,942,12256,1260,10
2,0.1500,0.0791,0.9527,0.1460,0.7929,907,10564,2952,45
3,0.2000,0.0879,0.8813,0.1599,0.6597,839,8705,4811,113
4,0.2500,0.0998,0.7763,0.1768,0.5120,739,6668,6848,213
5,0.3000,0.1086,0.6387,0.1856,0.3870,608,4991,8525,344
6,0.3500,0.1140,0.4737,0.1838,0.2734,451,3505,10011,501
7,0.4000,0.1206,0.3519,0.1796,0.1920,335,2443,11073,617
8,0.4500,0.1184,0.2405,0.1587,0.1337,229,1705,11811,723
9,0.5000,0.1121,0.1597,0.1317,0.0937,152,1204,12312,800


## 9. Calibration

In [10]:
calibration_df = calibration_table(
    y_test,
    test_probability,
    n_bins=10,
)

display(calibration_df)

print(
    "Mean absolute calibration gap:",
    f"{calibration_df['calibration_gap'].abs().mean():.4f}",
)

,mean_predicted_probability,observed_late_rate,calibration_gap
0,0.0752,0.0104,-0.0649
1,0.1261,0.0180,-0.1082
2,0.1665,0.0339,-0.1326
3,0.2034,0.0394,-0.1640
4,0.2373,0.0560,-0.1813
5,0.2742,0.0705,-0.2038
6,0.3155,0.0802,-0.2353
7,0.3639,0.1064,-0.2575
8,0.4366,0.1292,-0.3074
9,0.5937,0.1140,-0.4797


Mean absolute calibration gap: 0.2135


Calibration answers a different question from ranking:

- PR-AUC / ROC-AUC → can the model rank risky orders?
- Calibration → does a predicted probability such as 0.30 correspond to roughly a 30% observed risk?

Do not recalibrate on the observed test set.

## 10. Segment analysis

In [11]:
segment_specs = [
    "customer_state",
    "distance_band",
    "promise_band",
    "approval_lag_band",
    "prediction_month",
]

segment_results = {}

for segment_column in segment_specs:
    result = segment_metrics(
        test_scored,
        segment_column=segment_column,
        target_column=TARGET_COLUMN,
        probability_column="probability",
        minimum_rows=100,
        minimum_positives=10,
    )
    segment_results[segment_column] = result

    print("\nSEGMENT:", segment_column)
    display(result)


SEGMENT: customer_state


,segment,rows,positives,prevalence,pr_auc,pr_auc_lift,roc_auc,brier_score,log_loss
0,SP,6747,641,0.0950,0.2095,2.2057,0.7114,0.1080,0.3737
1,RJ,1627,88,0.0541,0.1394,2.5770,0.7370,0.1100,0.3718
2,MG,1581,33,0.0209,0.0601,2.8783,0.6952,0.0703,0.2824
3,PR,742,23,0.0310,0.1197,3.8608,0.7257,0.0530,0.2323
4,RS,719,24,0.0334,0.0582,1.7422,0.6620,0.1085,0.3731
5,SC,462,16,0.0346,0.1586,4.5800,0.6766,0.1087,0.3819
6,BA,457,20,0.0438,0.1651,3.7721,0.6605,0.3154,0.8306
7,DF,355,14,0.0394,0.0673,1.7061,0.5318,0.1327,0.4355
8,GO,275,11,0.0400,0.2314,5.7842,0.7014,0.1185,0.4016
9,ES,273,13,0.0476,0.2151,4.5172,0.7648,0.1923,0.5642



SEGMENT: distance_band


,segment,rows,positives,prevalence,pr_auc,pr_auc_lift,roc_auc,brier_score,log_loss
0,<=250 km,4752,586,0.1233,0.2258,1.8311,0.6911,0.1215,0.4058
1,250-500 km,3942,121,0.0307,0.1075,3.5024,0.7109,0.0754,0.2963
2,500-1000 km,3548,133,0.0375,0.1151,3.0714,0.7576,0.1034,0.3579
3,1000-2000 km,1382,57,0.0412,0.0915,2.2191,0.6955,0.2145,0.6069
4,>2000 km,786,49,0.0623,0.1348,2.1626,0.7524,0.2246,0.6330



SEGMENT: promise_band


,segment,rows,positives,prevalence,pr_auc,pr_auc_lift,roc_auc,brier_score,log_loss
0,10-20 days,5803,244,0.0420,0.0872,2.0741,0.6659,0.1205,0.4078
1,20-30 days,4212,108,0.0256,0.0510,1.9907,0.6558,0.1064,0.3574
2,<=10 days,2964,589,0.1987,0.2628,1.3223,0.5800,0.1755,0.5353
3,30-40 days,1244,11,0.0088,0.0285,3.2261,0.7387,0.0403,0.1942



SEGMENT: approval_lag_band


,segment,rows,positives,prevalence,pr_auc,pr_auc_lift,roc_auc,brier_score,log_loss
0,<=1 h,8983,542,0.0603,0.0980,1.6246,0.6720,0.1079,0.3691
1,6-24 h,2306,156,0.0676,0.1029,1.5204,0.6653,0.1315,0.4271
2,24-72 h,2099,179,0.0853,0.1533,1.7976,0.7107,0.1480,0.4642
3,1-6 h,758,56,0.0739,0.1010,1.3671,0.6593,0.1231,0.4049
4,>72 h,322,19,0.0590,0.1349,2.2857,0.6939,0.1396,0.4449



SEGMENT: prediction_month


,segment,rows,positives,prevalence,pr_auc,pr_auc_lift,roc_auc,brier_score,log_loss
0,2018-08,6498,685,0.1054,0.1326,1.2580,0.6171,0.1476,0.4650
1,2018-07,6047,248,0.0410,0.0894,2.1788,0.6627,0.1113,0.3796
2,2018-06,1923,19,0.0099,0.0428,4.3319,0.7956,0.0462,0.2124


## 11. Development → observed-test feature drift

Population Stability Index (PSI) is used here as a compact diagnostic.

Interpretation is heuristic:

- `< 0.10` → low;
- `0.10–0.25` → moderate;
- `> 0.25` → high.

PSI alone does not prove model degradation; compare it with temporal performance and segment results.

In [12]:
drift_df = drift_summary(
    development_df,
    test_df,
    numeric_features=NUMERIC_FEATURES,
    categorical_features=CATEGORICAL_FEATURES,
)

display(drift_df.head(25))

,feature,feature_type,psi,reference_missing_pct,comparison_missing_pct,drift_flag
0,purchase_month,numeric,11.2045,0.0000,0.0000,high
1,purchase_month_cos,numeric,10.8140,0.0000,0.0000,high
2,purchase_month_sin,numeric,9.4534,0.0000,0.0000,high
3,purchase_year,numeric,7.4307,0.0000,0.0000,high
4,promised_delivery_days,numeric,0.5809,0.0000,0.0000,high
5,total_freight,numeric,0.2381,0.0000,0.0000,moderate
6,dominant_product_category,categorical,0.1691,0.0000,0.0000,moderate
7,primary_payment_type,categorical,0.0362,0.0012,0.0000,low
8,mean_product_volume_cm3,numeric,0.0280,0.0195,0.0000,low
9,approval_lag_hours,numeric,0.0255,0.0000,0.0000,low


## 12. XGBoost native importance

In [13]:
native_importance_df = native_xgboost_importance(
    model
)

display(native_importance_df.head(30))

,feature,importance
0,numeric__purchase_month_cos,0.0726
1,categorical__customer_state_SP,0.0599
2,numeric__same_state_seller_share,0.0590
3,categorical__customer_state_RJ,0.0405
4,categorical__customer_state_MG,0.0240
5,numeric__purchase_year,0.0232
6,numeric__min_distance_km,0.0210
7,numeric__all_sellers_same_state,0.0205
8,numeric__purchase_month,0.0202
9,numeric__promised_delivery_days,0.0193


Native tree importance is model-specific and can be biased toward features that offer many possible split points.

Use permutation importance and SHAP as complementary views rather than treating native importance as causal importance.

## 13. Raw-feature permutation importance

In [14]:
permutation_df = raw_permutation_importance(
    model,
    X_test,
    y_test,
    n_repeats=5,
    random_state=42,
    n_jobs=-1,
)

display(permutation_df.head(30))

,feature,importance_mean,importance_std
0,promised_delivery_days,0.0501,0.0007
1,approval_lag_hours,0.0039,0.0009
2,dominant_product_category,0.0014,0.0003
3,purchase_weekday,0.0011,0.0002
4,seller_count,0.0010,0.0006
5,freight_ratio,0.0006,0.0004
6,total_product_weight_g,0.0005,0.0003
7,mean_product_weight_g,0.0004,0.0003
8,purchase_weekday_sin,0.0002,0.0001
9,mean_product_volume_cm3,0.0002,0.0004


Permutation importance measures the drop in observed-test Average Precision after each raw feature is shuffled.

Because the test set is already an observed diagnostic set, use this for explanation—not for feature selection or another tuning round.

## 14. SHAP global importance

In [15]:
try:
    (
        shap_importance_df,
        shap_values,
        shap_feature_names,
    ) = shap_global_importance(
        model,
        X_test,
        sample_size=2_000,
        random_state=42,
    )

    display(shap_importance_df.head(30))
except ImportError as exc:
    print(exc)

,feature,mean_abs_shap
0,numeric__purchase_month_cos,0.8796
1,numeric__promised_delivery_days,0.6083
2,categorical__customer_state_SP,0.1637
3,numeric__min_distance_km,0.1551
4,numeric__same_state_seller_share,0.1295
5,numeric__purchase_month,0.1161
6,numeric__approval_lag_hours,0.0948
7,numeric__mean_distance_km,0.0838
8,categorical__customer_state_MG,0.0788
9,numeric__purchase_year,0.0699


## 15. Export all Phase 5 artifacts

The reusable CLI performs the full export:

`python -m src.models.analyze`

If SHAP is not installed:

`python -m src.models.analyze --skip-shap`

# 16. Final interpretation checklist

After Run All / CLI execution, document the following from actual outputs:

1. OOF PR-AUC, prevalence and PR-AUC lift.
2. Observed-test PR-AUC, prevalence and lift.
3. Whether ROC-AUC and normalized PR-AUC remain stable through time.
4. Which operating threshold is acceptable for the intervention workflow.
5. Whether predicted probabilities are calibrated.
6. Which customer/geographic/time segments perform worst.
7. Which raw features show the strongest development → test drift.
8. Which features are important consistently across:
   - native XGBoost importance;
   - permutation importance;
   - SHAP.
9. Model limitations:
   - historical Olist data;
   - temporal distribution shift;
   - limited operational/logistics features;
   - observed rather than pristine final test;
   - threshold depends on intervention costs.

## Phase 5 completion criteria

Phase 5 is complete when:

- evaluation artifacts are exported;
- observed-test diagnostics are documented without further tuning;
- temporal robustness is explained;
- business threshold trade-offs are explicit;
- calibration is assessed;
- important failure segments are identified;
- feature importance and drift are documented;
- model limitations are written transparently.

The next stage is:

**Phase 6 — Productionization & Deployment**